- A decision tree is a flowchart of yes/no questions that ends in a prediction. You have seen the idea already: Day 7’s DummyClassifier asks no questions, and logistic regression multiplies weights. A tree is different because it learns which questions to ask, and in what order.

# Impurity: how mixed is a node?
- A good question sends mostly-survivors to one side and mostly-died to the other. A node is pure if it holds one class only, and impure if the classes are mixed. We measure the mix with a number.

# Information gain: how a split is chosen
- The tree tries many possible questions at each node and keeps the one with the biggest information gain

# Prediction and the overfitting danger
- A new row follows the questions down to a leaf. The leaf predicts the majority class of the training rows that landed there.
- predict_proba gives the share of each class in the leaf.
- If you do not stop it, a tree keeps splitting until every leaf is pure. That means it memorises the training rows.
- Controls that stop it: max_depth, min_samples_split, min_samples_leaf.

In [3]:
import numpy as np

def gini(counts):
    p = np.array(counts) / sum(counts)
    return 1 - np.sum(p ** 2)

def entropy(counts):
    p = np.array(counts) / sum(counts)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def split_gain(parent, left, right, impurity):
    n = sum(parent)
    child = sum(left) / n * impurity(left) + sum(right) / n * impurity(right)
    return impurity(parent) - child

In [4]:
parent = [6, 4]
print("Gini parent", round(gini(parent), 4), "Entropy parent", round(entropy(parent), 4))

female, male = [5, 0], [1, 4]
print("A is_female:", round(gini(female), 4), round(gini(male), 4),
      "gain gini", round(split_gain(parent, female, male, gini), 4),
      "gain entropy", round(split_gain(parent, female, male, entropy), 4))

c3, other = [2, 3], [4, 1]           # split B: pclass == 3
print("B pclass3:", round(gini(c3), 4), round(gini(other), 4),
      "gain gini", round(split_gain(parent, c3, other, gini), 4),
      "gain entropy", round(split_gain(parent, c3, other, entropy), 4))

Gini parent 0.48 Entropy parent 0.971
A is_female: 0.0 0.32 gain gini 0.32 gain entropy 0.61
B pclass3: 0.48 0.32 gain gini 0.08 gain entropy 0.1245


In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, export_text

In [6]:
df = sns.load_dataset("titanic")
data = df[["survived", "pclass", "sex", "age", "fare", "sibsp", "parch"]].copy()
data["is_female"] = (data["sex"] == "female").astype(int)
X = data[["pclass", "is_female", "age", "fare", "sibsp", "parch"]]
y = data["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.2, random_state=42, stratify=y
)
imp = SimpleImputer(strategy="median").fit(X_train)
Xtr = pd.DataFrame(imp.transform(X_train), columns=X.columns, index=X_train.index)
Xte = pd.DataFrame(imp.transform(X_test), columns=X.columns, index=X_test.index)

In [7]:
# 1. depth table
for d in (1, 2, 3, 4, 5, 8, None):
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xtr, y_train)
    print(d, t.get_depth(), t.get_n_leaves(),
          round(t.score(Xtr, y_train), 4), round(t.score(Xte, y_test), 4))

1 1 2 0.736 0.7994
2 2 4 0.736 0.7994
3 3 7 0.8146 0.7335
4 4 12 0.8539 0.8065
5 5 19 0.8596 0.763
8 8 36 0.9382 0.7896
None 14 53 0.9719 0.7588


In [8]:
# 2. read a small tree
t2 = DecisionTreeClassifier(max_depth=2, random_state=42).fit(Xtr, y_train)
print(export_text(t2, feature_names=list(X.columns), show_weights=True))

|--- is_female <= 0.50
|   |--- fare <= 26.14
|   |   |--- weights: [70.00, 8.00] class: 0
|   |--- fare >  26.14
|   |   |--- weights: [18.00, 17.00] class: 0
|--- is_female >  0.50
|   |--- pclass <= 1.50
|   |   |--- weights: [0.00, 17.00] class: 1
|   |--- pclass >  1.50
|   |   |--- weights: [22.00, 26.00] class: 1



In [9]:
# 3. which features did the tree use?
t3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xtr, y_train)
print(pd.Series(t3.feature_importances_, index=X.columns).round(3))

pclass       0.143
is_female    0.432
age          0.044
fare         0.246
sibsp        0.135
parch        0.000
dtype: float64


In [10]:
from sklearn.pipeline import Pipeline

res = {3: [], None: []}
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    for d in res:
        p = Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("tree", DecisionTreeClassifier(max_depth=d, random_state=42))]).fit(Xa, ya)
        res[d].append((p.score(Xa, ya), p.score(Xb, yb)))

for d, v in res.items():
    v = np.array(v)
    print(d, "train mean", v[:, 0].mean().round(4), "test mean", v[:, 1].mean().round(4),
          "test std", v[:, 1].std().round(4), "min", v[:, 1].min().round(4), "max", v[:, 1].max().round(4))

3 train mean 0.8248 test mean 0.8254 test std 0.015 min 0.8045 max 0.8659
None train mean 0.9818 test mean 0.7813 test std 0.0317 min 0.7039 max 0.838


# Bootstrap sample
- Pick n rows with replacement from n training rows. Some rows appear 2 or 3 times, and some never appear.
- It gives each tree a slightly different dataset, so the trees disagree in useful ways.

# Out-of-bag (OOB) rows
- The rows a tree never saw (about 36.8%).
- They work as a free validation set. Each row is scored only by the trees that never saw it.

# Bagging
- Bootstrap + Aggregate: train one tree per bootstrap sample, then vote (classification) or average (regression).
- Averaging many unstable models reduces variance.

# Random feature selection
- At each split, a tree may only consider a random subset of features (max_features; the default for classification is √(number of features)).
- Without it, every tree would pick is_female first and the trees would look alike. Different trees make different mistakes, and voting needs that.

# Random Forest
- Bagging + random feature selection, using deep trees.
- Many diverse, individually overfit trees, averaged into a stable answer.

In [16]:
import numpy as np

rng = np.random.default_rng(0)
rows = np.arange(10)
print("Bootstrap of 0..9:", np.sort(rng.choice(rows, size=10, replace=True)))

n = len(Xtr)
shares = []
for i in range(1000):
    s = rng.choice(n, size=n, replace=True)
    shares.append(len(np.unique(s)) / n)
print("n", n, "mean unique share", round(np.mean(shares), 4), 
      "oob share", round(1 - np.mean(shares), 4))

Bootstrap of 0..9: [0 0 0 1 2 3 5 6 8 8]
n 178 mean unique share 0.635 oob share 0.365


In [17]:
from sklearn.tree import DecisionTreeClassifier

Xtr_r = Xtr.reset_index(drop=True)
ytr_r = y_train.reset_index(drop=True)

rng = np.random.default_rng(42)
trees = []
for i in range(25):
    idx = rng.choice(n, size=n, replace=True)          # one bootstrap sample
    t = DecisionTreeClassifier(random_state=i).fit(Xtr_r.iloc[idx], ytr_r.iloc[idx])
    trees.append(t)

single = [round(t.score(Xte, y_test), 4) for t in trees]
print("single trees test: mean", round(np.mean(single), 4), "min", min(single), "max", max(single))

votes = np.array([t.predict(Xte) for t in trees])      # shape (25, 179)
pred = (votes.mean(axis=0) >= 0.5).astype(int)         # majority vote
print("bagged vote test acc", round((pred == y_test.values).mean(), 4))
print("first test passenger:", votes[:, 0].sum(), "of 25 say survive; true", y_test.values[0])

single trees test: mean 0.7314 min 0.6718 max 0.7896
bagged vote test acc 0.7924
first test passenger: 10 of 25 say survive; true 1


In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42, oob_score=True)
rf.fit(X_train, y_train)
print("RF200 train", round(rf.score(Xtr, y_train), 4), 
      "oob", round(rf.oob_score_, 4), "test", round(rf.score(Xte, y_test), 4))
print(pd.Series(rf.feature_importances_, index=X.columns).round(3))

tree = DecisionTreeClassifier(random_state=42).fit(Xtr, y_train)
print("single unlimited tree test", round(tree.score(Xte, y_test), 4))

for mf in (1, 2, 3, 6):
    r = RandomForestClassifier(n_estimators=200, max_features=mf, random_state=42, oob_score=True).fit(Xtr, y_train)
    print("max_features", mf, "oob", round(r.oob_score_, 4), "test", round(r.score(Xte, y_test), 4))

RF200 train 0.927 oob 0.7022 test 0.805
pclass       0.102
is_female    0.176
age          0.254
fare         0.339
sibsp        0.068
parch        0.060
dtype: float64
single unlimited tree test 0.7588
max_features 1 oob 0.7528 test 0.8121
max_features 2 oob 0.7247 test 0.8036
max_features 3 oob 0.736 test 0.8107
max_features 6 oob 0.764 test 0.791


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

models = {
    "tree_unlimited": lambda: DecisionTreeClassifier(random_state=42),
    "tree_depth3": lambda: DecisionTreeClassifier(max_depth=3, random_state=42),
    "forest_200": lambda: RandomForestClassifier(n_estimators=200, random_state=42),
    "forest_leaf5": lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42 )
}
scores = {k: [] for k in models}

for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    imp = SimpleImputer(strategy="median").fit(Xa)
    A, B = imp.transform(Xa), imp.transform(Xb)
    for name, make in models.items():
        scores[name].append(make().fit(A, ya).score(B, yb))

for name, v in scores.items():
    print(name, "mean", round(np.mean(v), 4), "std", round(np.std(v), 4),
          "min", round(min(v), 4), "max", round(max(v), 4))

wins = sum(f > t for f, t in zip(scores["forest_200"], scores["tree_unlimited"]))
print("forest beats unlimited tree in", wins, "of 20  splits")

tree_unlimited mean 0.7813 std 0.0317 min 0.7039 max 0.838
tree_depth3 mean 0.8254 std 0.015 min 0.8045 max 0.8659
forest_200 mean 0.8265 std 0.0216 min 0.7821 max 0.8715
forest_leaf5 mean 0.8321 std 0.0148 min 0.7989 max 0.8547
forest beats unlimited tree in 18 of 20  splits


In [22]:
def run(**kw):
    r = RandomForestClassifier(random_state=42, **kw).fit(Xtr, y_train)
    return round(r.score(Xtr, y_train), 4), round(r.score(Xte, y_test), 4)

print("n_estimators  (train, test)")
for k in (1, 5, 25, 100, 300):
    print(" ", k, run(n_estimators=k))
print("max_depth")
for d in (1, 2, 3, 5, 8, None):
    print(" ", d, run(n_estimators=200, max_depth=d))
print("min_samples_split")
for m in (2, 10, 30, 80):
    print(" ", m, run(n_estimators=200, min_samples_split=m))
print("min_samples_leaf")
for m in (1, 3, 5, 10, 20):
    print(" ", m, run(n_estimators=200, min_samples_leaf=m))

n_estimators  (train, test)
  1 (0.882, 0.6802)
  5 (0.9382, 0.756)
  25 (0.9719, 0.7994)
  100 (0.9719, 0.805)
  300 (0.9719, 0.805)
max_depth
  1 (0.7303, 0.7125)
  2 (0.7472, 0.7419)
  3 (0.8202, 0.7686)
  5 (0.8989, 0.7994)
  8 (0.9438, 0.8093)
  None (0.9719, 0.8036)
min_samples_split
  2 (0.9719, 0.8036)
  10 (0.8933, 0.798)
  30 (0.8315, 0.7896)
  80 (0.7303, 0.7391)
min_samples_leaf
  1 (0.9719, 0.8036)
  3 (0.8764, 0.7938)
  5 (0.8427, 0.7826)
  10 (0.8258, 0.7714)
  20 (0.736, 0.6999)
